# FOSS4G 2026 Phase 1 Reproduction Notebook: 3-System Comparison (GraphRAG / Structured / Adaptive)

This notebook reproduces **Phase 1** (90 cases, Shibuya-only, 3-system comparison) of the FOSS4G 2026 Hiroshima Academic Track paper
*A Systematic Comparison of RAG Architectures for Geographic POI Question Answering Using OpenStreetMap Data*.

- Paper DOI: https://doi.org/10.5194/isprs-archives-L-4-W1-2026-219-2026
- Runtime: Google Colab (**GPU required**, same requirements as the Phase 2 notebook).
- If you only want to verify the **GPU-free parts** locally (test-case loading, POI data loading), see `pytest tests/test_smoke.py` in this repository.

Note: this notebook answers Japanese-language questions about Shibuya POIs, so the 90 test questions (`eval/test_cases_v2.py` + `eval/test_cases_graphrag.py`) are in Japanese, matching what was actually tested in the paper. Everything else here (comments, log output) is in English.

See also `docs/phase1_vs_phase2_test_design.md` for a comparison of how question difficulty was designed across Phase 1 (this notebook) and Phase 2.


# Adaptive RAG Evaluation Notebook

Evaluates an Adaptive RAG system that dynamically switches between GraphRAG and Structured RAG depending on the question type.

## Evaluation approach

- 90 test cases (55 for Structured RAG + 35 for GraphRAG)
- Compare 3 systems: GraphRAG, Structured RAG, Adaptive RAG
- Analyze routing accuracy by question type


## 1. Environment Setup


In [ ]:
# Environment check (Colab / local)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set the project path
    # Change this to match where you cloned/extracted this repo on Google Drive
    PROJECT_PATH = '/content/drive/MyDrive/foss4g2026-rag-poi-reproduction'
    sys.path.insert(0, f'{PROJECT_PATH}/src')
    
    # Install required packages
    !pip install -q chromadb sentence-transformers networkx
    !pip install -q transformers accelerate bitsandbytes
    !pip install -q langchain langchain-core langchain-community langchain-chroma
else:
    PROJECT_PATH = '..'
    sys.path.insert(0, f'{PROJECT_PATH}/src')

print(f"Project path: {PROJECT_PATH}")
print(f"Running in Colab: {IN_COLAB}")

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np

# Import test cases
sys.path.insert(0, f'{PROJECT_PATH}/eval')
from test_cases_v2 import TEST_CASES_V2
from test_cases_graphrag import GRAPHRAG_TEST_CASES

## 2. Set Up LLM and Embedding Model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load the LLM
model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load the embedding model
embed_model = SentenceTransformer('intfloat/multilingual-e5-base')

print("Models loaded successfully!")

## 3. Initialize the Adaptive RAG System


In [ ]:
import json
from adaptive_rag_system import AdaptiveRAGSystem, AdaptiveRAGResult
from structured_rag_system import analyze_question

# Load POI data
# This repro package ships the Shibuya POI data as data/poi_shibuya.json
POI_JSON_PATH = f"{PROJECT_PATH}/data/poi_shibuya.json"
with open(POI_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_pois = json.load(f)

print(f"Loaded {len(raw_pois)} POIs")

# Extract metadata into a flat POI list (used by StructuredRAGSystem)
all_pois = []
for poi in raw_pois:
    poi_data = poi['metadata'].copy()
    poi_data['content'] = poi['content']
    poi_data['name'] = poi_data.get('name', '不明')
    all_pois.append(poi_data)

# Build the vector store
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Embedding function
embedding_function = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-base',
    model_kwargs={'device': 'cuda'}
)

# Build documents
documents = [
    Document(page_content=poi['content'], metadata=poi['metadata'])
    for poi in raw_pois
]

# Index into ChromaDB
vectorstore = Chroma.from_documents(
    documents,
    embedding_function,
    collection_name="poi_collection"
)

print(f"Vectorstore created with {len(documents)} documents")

# Initialize the Adaptive RAG system
adaptive_rag = AdaptiveRAGSystem(
    model=model,
    tokenizer=tokenizer,
    vectorstore=vectorstore,
    all_pois=all_pois,
    include_extended_edges=True,
    verbose=True
)

print("Adaptive RAG System initialized!")

## 4. Analyze System Selection


In [ ]:
# Analyze system selection across all test cases
all_test_cases = []

# Structured-RAG-oriented test cases (use the `prompt` field)
for tc in TEST_CASES_V2:
    all_test_cases.append({
        "id": tc.id,
        "question": tc.prompt,  # TestCaseV2 uses the prompt field
        "category": tc.subcategory,
        "source": "structured",
        "expected_keywords": tc.expected_keywords
    })

# GraphRAG-oriented test cases (use the `question` field)
for tc in GRAPHRAG_TEST_CASES:
    all_test_cases.append({
        "id": tc.id,
        "question": tc.question,  # GraphRAGTestCase uses the question field
        "category": tc.category,
        "source": "graphrag",
        "expected_keywords": tc.expected_keywords
    })

print(f"Total test cases: {len(all_test_cases)}")
print(f"  - Structured RAG tests: {sum(1 for tc in all_test_cases if tc['source'] == 'structured')}")
print(f"  - GraphRAG tests: {sum(1 for tc in all_test_cases if tc['source'] == 'graphrag')}")

In [ ]:
# Predict the system selection for each test case
selection_predictions = []

for tc in all_test_cases:
    selected, reason = adaptive_rag.select_system(tc["question"])
    selection_predictions.append({
        "id": tc["id"],
        "question": tc["question"],
        "category": tc["category"],
        "source": tc["source"],
        "selected_system": selected,
        "selection_reason": reason
    })

# Selection statistics
graphrag_count = sum(1 for p in selection_predictions if p["selected_system"] == "GraphRAG")
structured_count = sum(1 for p in selection_predictions if p["selected_system"] == "StructuredRAG")

print(f"\nSystem Selection Statistics:")
print(f"  GraphRAG selected: {graphrag_count} ({graphrag_count/len(selection_predictions)*100:.1f}%)")
print(f"  StructuredRAG selected: {structured_count} ({structured_count/len(selection_predictions)*100:.1f}%)")

In [ ]:
# Selection distribution by category
category_selections = {}
for p in selection_predictions:
    cat = p["category"]
    if cat not in category_selections:
        category_selections[cat] = {"GraphRAG": 0, "StructuredRAG": 0}
    category_selections[cat][p["selected_system"]] += 1

print("\nSelection by Category:")
print("-" * 60)
for cat, counts in sorted(category_selections.items()):
    total = counts["GraphRAG"] + counts["StructuredRAG"]
    gr_pct = counts["GraphRAG"] / total * 100
    print(f"{cat:30s} GraphRAG: {counts['GraphRAG']:2d} ({gr_pct:5.1f}%) | Structured: {counts['StructuredRAG']:2d}")

## 5. 3-System Comparison Evaluation


In [ ]:
def calculate_keyword_score(response: str, expected_keywords: list) -> float:
    """Compute the keyword-matching score."""
    if not expected_keywords:
        return 1.0
    matched = sum(1 for kw in expected_keywords if kw.lower() in response.lower())
    return matched / len(expected_keywords)

def evaluate_system(system_name: str, test_cases: list, adaptive_rag: AdaptiveRAGSystem):
    """Run evaluation for the given system."""
    results = []
    
    for i, tc in enumerate(test_cases):
        print(f"\r[{system_name}] Evaluating {i+1}/{len(test_cases)}: {tc['id']}", end="")
        
        try:
            result = adaptive_rag.query_with_system(tc["question"], system_name)
            score = calculate_keyword_score(result.response, tc["expected_keywords"])
            
            results.append({
                "id": tc["id"],
                "category": tc["category"],
                "source": tc["source"],
                "score": score * 100,
                "response": result.response,
                "selected_system": result.selected_system,
                "selection_reason": result.selection_reason,
                "total_time": result.total_time
            })
        except Exception as e:
            print(f"\nError on {tc['id']}: {e}")
            results.append({
                "id": tc["id"],
                "category": tc["category"],
                "source": tc["source"],
                "score": 0,
                "response": f"Error: {e}",
                "selected_system": system_name,
                "selection_reason": "error",
                "total_time": 0
            })
    
    print(f"\n[{system_name}] Evaluation complete!")
    return results

In [ ]:
# GraphRAG evaluation
print("=" * 60)
print("GraphRAG Evaluation")
print("=" * 60)
graphrag_results = evaluate_system("GraphRAG", all_test_cases, adaptive_rag)

In [ ]:
# StructuredRAG evaluation
print("=" * 60)
print("StructuredRAG Evaluation")
print("=" * 60)
structured_results = evaluate_system("StructuredRAG", all_test_cases, adaptive_rag)

In [ ]:
# Adaptive RAG evaluation
print("=" * 60)
print("Adaptive RAG Evaluation")
print("=" * 60)
adaptive_results = evaluate_system("Adaptive", all_test_cases, adaptive_rag)

## 6. Result Analysis


In [ ]:
# Compute overall scores
def calc_stats(results):
    scores = [r["score"] for r in results]
    times = [r["total_time"] for r in results]
    return {
        "overall": np.mean(scores),
        "std": np.std(scores),
        "avg_time": np.mean(times)
    }

graphrag_stats = calc_stats(graphrag_results)
structured_stats = calc_stats(structured_results)
adaptive_stats = calc_stats(adaptive_results)

print("=" * 60)
print("Overall Results (90 test cases)")
print("=" * 60)
print(f"{'System':<20} {'Score':>10} {'Std':>10} {'Avg Time':>12}")
print("-" * 60)
print(f"{'GraphRAG':<20} {graphrag_stats['overall']:>9.1f}% {graphrag_stats['std']:>9.1f}% {graphrag_stats['avg_time']:>10.1f}s")
print(f"{'StructuredRAG':<20} {structured_stats['overall']:>9.1f}% {structured_stats['std']:>9.1f}% {structured_stats['avg_time']:>10.1f}s")
print(f"{'Adaptive RAG':<20} {adaptive_stats['overall']:>9.1f}% {adaptive_stats['std']:>9.1f}% {adaptive_stats['avg_time']:>10.1f}s")

In [ ]:
# Scores by test source
def calc_by_source(results, source):
    filtered = [r for r in results if r["source"] == source]
    return np.mean([r["score"] for r in filtered])

print("\nResults by Test Source:")
print("-" * 60)
print(f"{'System':<20} {'Structured (55)':>15} {'GraphRAG (35)':>15}")
print("-" * 60)
print(f"{'GraphRAG':<20} {calc_by_source(graphrag_results, 'structured'):>14.1f}% {calc_by_source(graphrag_results, 'graphrag'):>14.1f}%")
print(f"{'StructuredRAG':<20} {calc_by_source(structured_results, 'structured'):>14.1f}% {calc_by_source(structured_results, 'graphrag'):>14.1f}%")
print(f"{'Adaptive RAG':<20} {calc_by_source(adaptive_results, 'structured'):>14.1f}% {calc_by_source(adaptive_results, 'graphrag'):>14.1f}%")

In [ ]:
# Score comparison by category
def calc_by_category(results):
    categories = {}
    for r in results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r["score"])
    return {cat: np.mean(scores) for cat, scores in categories.items()}

graphrag_by_cat = calc_by_category(graphrag_results)
structured_by_cat = calc_by_category(structured_results)
adaptive_by_cat = calc_by_category(adaptive_results)

print("\nResults by Category:")
print("-" * 80)
print(f"{'Category':<25} {'GraphRAG':>12} {'Structured':>12} {'Adaptive':>12} {'Best':>12}")
print("-" * 80)

for cat in sorted(graphrag_by_cat.keys()):
    gr = graphrag_by_cat.get(cat, 0)
    st = structured_by_cat.get(cat, 0)
    ad = adaptive_by_cat.get(cat, 0)
    
    best = "Adaptive" if ad >= max(gr, st) else ("GraphRAG" if gr > st else "Structured")
    print(f"{cat:<25} {gr:>11.1f}% {st:>11.1f}% {ad:>11.1f}% {best:>12}")

## 7. Visualization


In [ ]:
# Overall comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall score
systems = ['GraphRAG', 'StructuredRAG', 'Adaptive RAG']
scores = [graphrag_stats['overall'], structured_stats['overall'], adaptive_stats['overall']]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = axes[0].bar(systems, scores, color=colors)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('Overall Score Comparison (90 test cases)')
axes[0].set_ylim(0, 100)

for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{score:.1f}%', ha='center', va='bottom', fontsize=12)

# By test source
x = np.arange(2)
width = 0.25

structured_scores = [calc_by_source(graphrag_results, 'structured'), 
                     calc_by_source(graphrag_results, 'graphrag')]
graphrag_scores_src = [calc_by_source(structured_results, 'structured'), 
                       calc_by_source(structured_results, 'graphrag')]
adaptive_scores = [calc_by_source(adaptive_results, 'structured'), 
                   calc_by_source(adaptive_results, 'graphrag')]

axes[1].bar(x - width, structured_scores, width, label='GraphRAG', color='#1f77b4')
axes[1].bar(x, graphrag_scores_src, width, label='StructuredRAG', color='#ff7f0e')
axes[1].bar(x + width, adaptive_scores, width, label='Adaptive RAG', color='#2ca02c')

axes[1].set_ylabel('Score (%)')
axes[1].set_title('Score by Test Source')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Structured Tests (55)', 'GraphRAG Tests (35)'])
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/adaptive_comparison_overall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Category comparison chart
categories = sorted(graphrag_by_cat.keys())
x = np.arange(len(categories))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))

gr_scores = [graphrag_by_cat.get(c, 0) for c in categories]
st_scores = [structured_by_cat.get(c, 0) for c in categories]
ad_scores = [adaptive_by_cat.get(c, 0) for c in categories]

bars1 = ax.bar(x - width, gr_scores, width, label='GraphRAG', color='#1f77b4')
bars2 = ax.bar(x, st_scores, width, label='StructuredRAG', color='#ff7f0e')
bars3 = ax.bar(x + width, ad_scores, width, label='Adaptive RAG', color='#2ca02c')

ax.set_ylabel('Score (%)')
ax.set_title('3 Systems Comparison by Category (90 test cases)')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 110)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/adaptive_comparison_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Adaptive RAG's system-selection distribution
adaptive_selections = {}
for r in adaptive_results:
    sys = r["selected_system"]
    if sys not in adaptive_selections:
        adaptive_selections[sys] = 0
    adaptive_selections[sys] += 1

fig, ax = plt.subplots(figsize=(8, 6))

systems = list(adaptive_selections.keys())
counts = list(adaptive_selections.values())
colors = ['#1f77b4' if s == 'GraphRAG' else '#ff7f0e' for s in systems]

bars = ax.bar(systems, counts, color=colors)
ax.set_ylabel('Number of Queries')
ax.set_title('Adaptive RAG: System Selection Distribution')

for bar, count in zip(bars, counts):
    pct = count / sum(counts) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{count} ({pct:.1f}%)', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig(f'{PROJECT_PATH}/results/adaptive_selection_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results


In [ ]:
# Save evaluation results as JSON
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

evaluation_results = {
    "metadata": {
        "timestamp": timestamp,
        "total_test_cases": len(all_test_cases),
        "model": "Qwen2.5-7B-Instruct (4bit)"
    },
    "summary": {
        "graphrag": graphrag_stats,
        "structured": structured_stats,
        "adaptive": adaptive_stats
    },
    "by_source": {
        "graphrag": {
            "on_structured_tests": calc_by_source(graphrag_results, 'structured'),
            "on_graphrag_tests": calc_by_source(graphrag_results, 'graphrag')
        },
        "structured": {
            "on_structured_tests": calc_by_source(structured_results, 'structured'),
            "on_graphrag_tests": calc_by_source(structured_results, 'graphrag')
        },
        "adaptive": {
            "on_structured_tests": calc_by_source(adaptive_results, 'structured'),
            "on_graphrag_tests": calc_by_source(adaptive_results, 'graphrag')
        }
    },
    "by_category": {
        "graphrag": graphrag_by_cat,
        "structured": structured_by_cat,
        "adaptive": adaptive_by_cat
    },
    "adaptive_selections": adaptive_selections,
    "detailed_results": {
        "graphrag": graphrag_results,
        "structured": structured_results,
        "adaptive": adaptive_results
    }
}

output_path = f"{PROJECT_PATH}/results/adaptive_evaluation_{timestamp}.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_path}")

## 9. Conclusion


In [ ]:
print("=" * 60)
print("Adaptive RAG Evaluation - Conclusion")
print("=" * 60)

print(f"\n【Overall Score (90 test cases)】")
print(f"  GraphRAG:     {graphrag_stats['overall']:.1f}%")
print(f"  StructuredRAG: {structured_stats['overall']:.1f}%")
print(f"  Adaptive RAG:  {adaptive_stats['overall']:.1f}%")

# Adaptive RAG's improvement
improvement_vs_graphrag = adaptive_stats['overall'] - graphrag_stats['overall']
improvement_vs_structured = adaptive_stats['overall'] - structured_stats['overall']

print(f"\n【Adaptive RAG Improvement】")
print(f"  vs GraphRAG:     {improvement_vs_graphrag:+.1f}%")
print(f"  vs StructuredRAG: {improvement_vs_structured:+.1f}%")

# Determine the best system
best_system = max(
    [("GraphRAG", graphrag_stats['overall']), 
     ("StructuredRAG", structured_stats['overall']), 
     ("Adaptive RAG", adaptive_stats['overall'])],
    key=lambda x: x[1]
)

print(f"\n【Best System】")
print(f"  {best_system[0]}: {best_system[1]:.1f}%")

# Adaptive RAG's selection distribution
print(f"\n【Adaptive RAG Selection Distribution】")
for sys, count in adaptive_selections.items():
    pct = count / len(adaptive_results) * 100
    print(f"  {sys}: {count} queries ({pct:.1f}%)")